In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Partition in Spark")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 03:30:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = (
    spark.range(1, 1001)
    .withColumnRenamed("id", "employee_id")
)

In [3]:
df.show()

[Stage 0:>                                                          (0 + 1) / 1]

+-----------+
|employee_id|
+-----------+
|          1|
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
|         10|
|         11|
|         12|
|         13|
|         14|
|         15|
|         16|
|         17|
|         18|
|         19|
|         20|
+-----------+
only showing top 20 rows



In [7]:
df.rdd.getNumPartitions()

24

In [8]:
from pyspark.sql import functions as F

df = (
    df
    .withColumn(
        "city",
        F.when((F.col("employee_id") % 4) == 0, "Delhi")
         .when((F.col("employee_id") % 4) == 1, "Mumbai")
         .when((F.col("employee_id") % 4) == 2, "Pune")
         .otherwise("Bangalore")
    )
    .withColumn(
        "department",
        F.when((F.col("employee_id") % 3) == 0, "Engineering")
         .when((F.col("employee_id") % 3) == 1, "HR")
         .otherwise("Sales")
    )
    .withColumn(
        "salary",
        (F.rand() * 100000 + 30000).cast("int")
    )
)

In [9]:
df.show(10)

+-----------+---------+-----------+------+
|employee_id|     city| department|salary|
+-----------+---------+-----------+------+
|          1|   Mumbai|         HR| 39744|
|          2|     Pune|      Sales| 75869|
|          3|Bangalore|Engineering| 62634|
|          4|    Delhi|         HR| 39628|
|          5|   Mumbai|      Sales|108775|
|          6|     Pune|Engineering|109890|
|          7|Bangalore|         HR| 78058|
|          8|    Delhi|      Sales|101320|
|          9|   Mumbai|Engineering| 43795|
|         10|     Pune|         HR| 34315|
+-----------+---------+-----------+------+
only showing top 10 rows



In [10]:
df.groupBy("city").count().show()

[Stage 2:============================>                           (12 + 12) / 24]

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|  250|
|   Mumbai|  250|
|     Pune|  250|
|    Delhi|  250|
+---------+-----+



In [12]:
# Write Partitioned Parquet
df.write.mode("overwrite").partitionBy("city").parquet("/data/processed/employees_partitioned")

In [13]:
# Inspect Physical Layout
'''Spark physically separated data into folders:
   city---→physical storage partition
'''
import os
os.listdir("/data/processed/employees_partitioned")

['city=Mumbai',
 'city=Delhi',
 'city=Pune',
 '._SUCCESS.crc',
 'city=Bangalore',
 '_SUCCESS']

In [14]:
df_delhi = spark.read.parquet("/data/processed/employees_partitioned").filter("city = 'Delhi'")
df_delhi.count()

250

In [15]:
df.groupBy("department").avg("salary").show()

[Stage 10:=====================================>                  (16 + 8) / 24]

+-----------+----------------+
| department|     avg(salary)|
+-----------+----------------+
|      Sales|81382.9039039039|
|Engineering|83034.8918918919|
|         HR|79251.4880239521|
+-----------+----------------+



In [16]:
df.cache()

DataFrame[employee_id: bigint, city: string, department: string, salary: int]

In [18]:
df_salary = df.groupBy("department").avg("salary")
df_salary.cache()
df_salary.show()

[Stage 18:=======================================>             (150 + 25) / 200]

+-----------+----------------+
| department|     avg(salary)|
+-----------+----------------+
|      Sales|81382.9039039039|
|Engineering|83034.8918918919|
|         HR|79251.4880239521|
+-----------+----------------+



In [19]:
df_salary.rdd.getNumPartitions()

200

In [20]:
df_salary.show()

+-----------+----------------+
| department|     avg(salary)|
+-----------+----------------+
|      Sales|81382.9039039039|
|Engineering|83034.8918918919|
|         HR|79251.4880239521|
+-----------+----------------+



In [21]:
## repartition() and  coalesce()
df.rdd.getNumPartitions()

24

In [22]:
df_repartitioned = df.repartition(8)

In [23]:
df_repartitioned.rdd.getNumPartitions()

8

In [25]:
''' counts the total rows, triggering a full shuffle. It forces an even distribution of data across partitions, 
which is useful for fixing data skew before saving, but it is an expensive operation
'''

df_repartitioned.count()

1000

In [26]:
df_coalesced = df.coalesce(4)

In [27]:
df_coalesced.rdd.getNumPartitions()

4

In [31]:
df_coalesced.count()

1000

In [32]:
df.repartition(8).write.mode("overwrite").parquet("/data/processed/repartitioned")

In [33]:
os.listdir("/data/processed/repartitioned")

['.part-00005-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 '.part-00004-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 'part-00003-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 'part-00002-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 'part-00000-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 '.part-00001-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 'part-00004-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 '._SUCCESS.crc',
 'part-00007-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 '_SUCCESS',
 '.part-00007-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 '.part-00002-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 'part-00006-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet',
 '.part-00003-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc',
 '.part-00006-7be649cb-791c-4032-981c-2d991d95abe5-c000.snappy.parquet.crc'

In [35]:
df.coalesce(2).write.mode("overwrite").parquet("/data/processed/coalesced")

In [36]:
os.listdir("/data/processed/coalesced")

['.part-00000-f4cb8691-b8ed-46d8-9cd0-265d4102dfb4-c000.snappy.parquet.crc',
 '.part-00001-f4cb8691-b8ed-46d8-9cd0-265d4102dfb4-c000.snappy.parquet.crc',
 '._SUCCESS.crc',
 '_SUCCESS',
 'part-00000-f4cb8691-b8ed-46d8-9cd0-265d4102dfb4-c000.snappy.parquet',
 'part-00001-f4cb8691-b8ed-46d8-9cd0-265d4102dfb4-c000.snappy.parquet']